In [38]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.model_selection import KFold
from sklearn.metrics import cohen_kappa_score

In [39]:
# -----------------------------
# 1️⃣ Cargar datos
# -----------------------------
folder = "/kaggle/input/petfinder-adoption-prediction"
train = pd.read_csv(f"{folder}/train/train.csv")
test = pd.read_csv(f"{folder}/test/test.csv")

target_col = "AdoptionSpeed"
X = train.set_index("PetID").drop(target_col, axis=1).select_dtypes(exclude="O")
y = train[target_col]
X_test = test.set_index("PetID")[X.columns]

In [40]:
# -----------------------------
# 2️⃣ Configurar K-Fold
# -----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=0)

all_preds = []       # predicciones de validación
all_test_preds = []  # predicciones de test
kappa_scores = []    # métricas

In [41]:
# -----------------------------
# 3️⃣ Entrenamiento por fold
# -----------------------------
for fold, (train_idx, valid_idx) in enumerate(kf.split(X), 1):
    print(f"\n=== Fold {fold} ===")

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_valid = y.iloc[valid_idx]

    model = LGBMClassifier()
    model.fit(X_train, y_train)

    # --- Validación ---
    valid_pred = pd.Series(
        model.predict(X_valid),
        index=X_valid.index,
        name=target_col
    )
    all_preds.append(valid_pred)

    kappa = cohen_kappa_score(y_valid, valid_pred, weights='quadratic')
    kappa_scores.append(kappa)
    print(f"Kappa cuadrático (validación): {kappa:.4f}")

    # --- Test ---
    test_pred = pd.Series(
        model.predict(X_test),
        index=X_test.index,
        name=target_col
    )
    all_test_preds.append(test_pred)


=== Fold 1 ===
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001479 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 438
[LightGBM] [Info] Number of data points in the train set: 11994, number of used features: 19
[LightGBM] [Info] Start training from score -3.581021
[LightGBM] [Info] Start training from score -1.589135
[LightGBM] [Info] Start training from score -1.309142
[LightGBM] [Info] Start training from score -1.527742
[LightGBM] [Info] Start training from score -1.269494
Kappa cuadrático (validación): 0.3316

=== Fold 2 ===
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001001 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 444
[LightGBM] [Info] Number of data points in the tr

In [43]:
# -----------------------------
# 4 Combinar predicciones de test por moda
# -----------------------------
test_preds_df = pd.concat(all_test_preds, axis=1)
final_test_pred = test_preds_df.mode(axis=1)[0]  # moda fila a fila

submission = pd.DataFrame({
    "PetID": final_test_pred.index,
    "AdoptionSpeed": final_test_pred.values
})
submission.to_csv("submission.csv", index=False)

print("\n✅ Archivo 'submission.csv' creado usando la moda de los folds.")
submission.head()


✅ Archivo 'submission.csv' creado usando la moda de los folds.


,PetID,AdoptionSpeed
0,e2dfc2935,4.0
1,f153b465f,2.0
2,3c90f3f54,2.0
3,e02abc8a3,4.0
4,09f0df7d1,4.0


In [44]:
# -----------------------------
# 4 Combinar predicciones de test por moda
# -----------------------------
test_preds_df = pd.concat(all_test_preds, axis=1)
final_test_pred = test_preds_df.mode(axis=1)[0]  # moda fila a fila

pd.concat(all_test_preds,axis=1).sort_index()

,AdoptionSpeed,AdoptionSpeed,AdoptionSpeed,AdoptionSpeed,AdoptionSpeed
PetID,,,,,
000aa306a,4,4,4,4,4
001630d8d,4,4,4,4,4
002089611,2,1,2,2,2
002efc654,4,4,4,4,4
0061e61ce,1,1,4,2,2
...,...,...,...,...,...
ffb8f5704,2,2,2,2,1
ffbbed680,4,4,4,4,4
ffc5dccdd,3,3,3,3,3
